In [1]:
from src.dataset import load_cifar10

In [2]:
(X_train, Y_train), (X_test, Y_test) = load_cifar10()

In [3]:
from src.model import initial_resnet

In [11]:
data_aug_model = initial_resnet()

In [12]:
data_aug_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)    │ (None, 32, 32, 3)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential (Sequential)       │ (None, 32, 32, 3)         │               0 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_14 (Conv2D)            │ (None, 32, 32, 24)        │             648 │ sequential[1][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_14        │ (None, 32, 32, 24)        │              96 │ conv2d_14[0][0]            │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_11 (ReLU)               │ (None, 32, 32, 24)        │               0 │ batch_normalization_14[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_1               │ (None, 16, 16, 24)        │               0 │ re_lu_11[0][0]             │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_16 (Conv2D)            │ (None, 16, 16, 48)        │          10,368 │ max_pooling2d_1[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_16        │ (None, 16, 16, 48)        │             192 │ conv2d_16[0][0]            │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_12 (ReLU)               │ (None, 16, 16, 48)        │               0 │ batch_normalization_16[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_17 (Conv2D)            │ (None, 16, 16, 48)        │          20,736 │ re_lu_12[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_15 (Conv2D)            │ (None, 16, 16, 48)        │           1,152 │ max_pooling2d_1[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_17        │ (None, 16, 16, 48)        │             192 │ conv2d_17[0][0]            │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_15        │ (None, 16, 16, 48)        │             192 │ conv2d_15[0][0]            │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_5 (Add)                   │ (None, 16, 16, 48)        │               0 │ batch_normalization_17[0]… │
│                               │                           │                 │ batch_normalization_15[0]… │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 1,515,826 (5.78 MB)

 Trainable params: 1,512,610 (5.77 MB)

 Non-trainable params: 3,216 (12.56 KB)

In [4]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
early_stop = EarlyStopping(
    monitor='val_loss',       
    patience=5,               
    restore_best_weights=True )

checkpoint = ModelCheckpoint(
    filepath='models/width_resnet.keras', 
    monitor='val_loss',      
    save_best_only=True,      
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=3, 
    min_lr=1e-5,
    verbose=1
)

In [8]:
import tensorflow as tf

In [13]:
data_aug_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [11]:
history = base_model.fit(
    X_train, Y_train,
    epochs=50,
    batch_size = 64,
    validation_split= 0.2,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.8253 - loss: 0.4996
Epoch 1: val_loss did not improve from 0.86204
625/625 ━━━━━━━━━━━━━━━━━━━━ 107s 170ms/step - accuracy: 0.8176 - loss: 0.5216 - val_accuracy: 0.6422 - val_loss: 1.1592 - learning_rate: 0.0010
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.8678 - loss: 0.3815
Epoch 2: val_loss did not improve from 0.86204
625/625 ━━━━━━━━━━━━━━━━━━━━ 106s 170ms/step - accuracy: 0.8575 - loss: 0.4034 - val_accuracy: 0.6935 - val_loss: 1.0122 - learning_rate: 0.0010
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.8984 - loss: 0.2923
Epoch 3: val_loss did not improve from 0.86204

Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
625/625 ━━━━━━━━━━━━━━━━━━━━ 107s 172ms/step - accuracy: 0.8885 - loss: 0.3167 - val_accuracy: 0.7185 - val_loss: 0.9188 - learning_rate: 0.0010
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 0.9559 - loss: 0.

In [12]:
from src.history import *

In [14]:
save_history(history, "history/baseline_resnet.pkl")

In [14]:
history1 = data_aug_model.fit(
    X_train, Y_train,
    epochs=50,
    batch_size = 64,
    validation_split= 0.2,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - accuracy: 0.4179 - loss: 1.6136
Epoch 1: val_loss improved from None to 1.32227, saving model to models/with_data_aug_resnet.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 114s 173ms/step - accuracy: 0.4884 - loss: 1.4157 - val_accuracy: 0.5210 - val_loss: 1.3223 - learning_rate: 0.0010
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 0.5902 - loss: 1.1477
Epoch 2: val_loss did not improve from 1.32227
625/625 ━━━━━━━━━━━━━━━━━━━━ 114s 182ms/step - accuracy: 0.6041 - loss: 1.1085 - val_accuracy: 0.5580 - val_loss: 1.4391 - learning_rate: 0.0010
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.6454 - loss: 0.9930
Epoch 3: val_loss improved from 1.32227 to 1.03021, saving model to models/with_data_aug_resnet.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 111s 177ms/step - accuracy: 0.6569 - loss: 0.9664 - val_accuracy: 0.6411 - val_loss: 1.0302 - learning_rate: 0.0010
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step

In [19]:
save_history(history1, "history/data_aug_resnet.pkl")

In [8]:
dropout_model = initial_resnet()

In [9]:
dropout_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 32, 32, 3)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential (Sequential)       │ (None, 32, 32, 3)         │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 32, 32, 24)        │             648 │ sequential[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 32, 32, 24)        │              96 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu (ReLU)                  │ (None, 32, 32, 24)        │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 16, 16, 24)        │               0 │ re_lu[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 16, 16, 48)        │          10,368 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 16, 16, 48)        │             192 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_1 (ReLU)                │ (None, 16, 16, 48)        │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 16, 16, 48)        │          20,736 │ re_lu_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 16, 16, 48)        │           1,152 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 16, 16, 48)        │             192 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 16, 16, 48)        │             192 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 16, 16, 48)        │               0 │ batch_normalization_3[0][… │
│                               │                           │                 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_2 (ReLU)                │ (None, 16, 16, 48)        │               

 Total params: 1,515,826 (5.78 MB)

 Trainable params: 1,512,610 (5.77 MB)

 Non-trainable params: 3,216 (12.56 KB)

In [10]:
dropout_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [11]:
history2 = dropout_model.fit(
    X_train, Y_train,
    epochs=50,
    batch_size = 64,
    validation_split= 0.2,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 0.4081 - loss: 1.6659
Epoch 1: val_loss improved from None to 1.39772, saving model to models/dropout_resnet.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 121s 184ms/step - accuracy: 0.4796 - loss: 1.4495 - val_accuracy: 0.5306 - val_loss: 1.3977 - learning_rate: 0.0010
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - accuracy: 0.5878 - loss: 1.1608
Epoch 2: val_loss improved from 1.39772 to 1.18019, saving model to models/dropout_resnet.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 118s 189ms/step - accuracy: 0.6038 - loss: 1.1219 - val_accuracy: 0.6060 - val_loss: 1.1802 - learning_rate: 0.0010
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.6512 - loss: 0.9869
Epoch 3: val_loss improved from 1.18019 to 0.97656, saving model to models/dropout_resnet.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 114s 183ms/step - accuracy: 0.6556 - loss: 0.9733 - val_accuracy: 0.6717 - val_loss: 0.9766 - learning_rate: 0.0010
Epoch 4/50
625/6

In [13]:
save_history(history2, "history/dropout_resnet.pkl")

In [5]:
width_model = initial_resnet()

In [6]:
width_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 32, 32, 3)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential (Sequential)       │ (None, 32, 32, 3)         │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 32, 32, 24)        │             648 │ sequential[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 32, 32, 24)        │              96 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu (ReLU)                  │ (None, 32, 32, 24)        │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 16, 16, 24)        │               0 │ re_lu[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 16, 16, 48)        │          10,368 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 16, 16, 48)        │             192 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_1 (ReLU)                │ (None, 16, 16, 48)        │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 16, 16, 48)        │          20,736 │ re_lu_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 16, 16, 48)        │           1,152 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 16, 16, 48)        │             192 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 16, 16, 48)        │             192 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 16, 16, 48)        │               0 │ batch_normalization_3[0][… │
│                               │                           │                 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_2 (ReLU)                │ (None, 16, 16, 48)        │               

 Total params: 1,180,018 (4.50 MB)

 Trainable params: 1,177,122 (4.49 MB)

 Non-trainable params: 2,896 (11.31 KB)

In [9]:
width_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [10]:
history3 = width_model.fit(
    X_train, Y_train,
    epochs=50,
    batch_size = 64,
    validation_split= 0.2,
    callbacks=[early_stop, checkpoint, reduce_lr]
)

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 0.4085 - loss: 1.6515
Epoch 1: val_loss improved from None to 1.79863, saving model to models/width_resnet.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 108s 162ms/step - accuracy: 0.4803 - loss: 1.4407 - val_accuracy: 0.4384 - val_loss: 1.7986 - learning_rate: 0.0010
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 0.5823 - loss: 1.1832
Epoch 2: val_loss improved from 1.79863 to 1.31399, saving model to models/width_resnet.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 105s 168ms/step - accuracy: 0.5982 - loss: 1.1345 - val_accuracy: 0.5695 - val_loss: 1.3140 - learning_rate: 0.0010
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.6478 - loss: 1.0074
Epoch 3: val_loss improved from 1.31399 to 1.29663, saving model to models/width_resnet.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 101s 162ms/step - accuracy: 0.6558 - loss: 0.9778 - val_accuracy: 0.5795 - val_loss: 1.2966 - learning_rate: 0.0010
Epoch 4/50
625/625 ━━━

In [13]:
save_history(history3, "history/witdth_resnet.pkl")